# cleaning.ipynb

Combine and reformat raw data for use in modeling further in the pipeline.

In [95]:
# Globals for this file, need to be updated once
# a new season, or sport is added
SPORTS_SEASONS = {"nfl":[2025], 
                  "mlb":[2025], 
                  "nba":[2025], 
                  "nhl":[2025]}

STATES = ["away", "home"]

import csv
import sys
import pandas as pd

sys.path.append("..")
from src.pricing.devig import devig_two_way, american_odds_to_prob

In [ ]:
def read_season_data_file(sport: str, season: int) -> pd.DataFrame:
    """Read in .csv file from data directory and return dataframe containing the data"""
    if sport not in SPORTS_SEASONS.keys() or season not in SPORTS_SEASONS[sport]:
        raise ValueError("Dataset not currently available.")

    df = pd.read_csv(f"../data/{sport}_{season}_historical.csv")

    return df

def clean_data(df: pd.DataFrame, de_vig_strat: str = "cheung") -> pd.DataFrame:
     """Clean datasets for analysis"""
     # Drop useless columns
     df = df.drop(columns=["game_id",
                      "kalshi_home_ticker",
                      "kalshi_away_ticker", 
                      "kalshi_home_candle_end_ts", 
                      "kalshi_away_candle_end_ts",
                      "sportsbook_bookmaker"])

     df = df.dropna(subset=["kalshi_home_yes_bid_close", 
                            "kalshi_home_yes_ask_close",
                            "kalshi_away_yes_bid_close", 
                            "kalshi_away_yes_ask_close",
                            "kalshi_home_open_interest", 
                            "kalshi_away_open_interest",
                            "sportsbook_home_odds_american", 
                            "sportsbook_away_odds_american",])

     df["total_volume"] = df[f"kalshi_{STATES[0]}_volume"] + df[f"kalshi_{STATES[1]}_volume"]

    # Compute Kalshi implied prob
     for state in STATES:
          df[f"Kalshi_{state}_prob"] = (df[f"kalshi_{state}_yes_bid_close"] + df[f"kalshi_{state}_yes_ask_close"]) / 2
          df = df.drop(columns=[f"kalshi_{state}_close_price",
                                f"kalshi_{state}_yes_bid_close",
                                f"kalshi_{state}_yes_ask_close"])

     
     df[[f"sportsbook_{STATES[0]}_prob", f"sportsbook_{STATES[1]}_prob"]] = df.apply(
          lambda row: devig_two_way(
               american_odds_to_prob(row[f"sportsbook_{STATES[0]}_odds_american"]),
               american_odds_to_prob(row[f"sportsbook_{STATES[1]}_odds_american"]),
               de_vig_strat=de_vig_strat,
          ),
          axis=1,
          result_type="expand",
     )

     df = df.drop(columns=[f"sportsbook_{STATES[0]}_odds_american",
                           f"sportsbook_{STATES[1]}_odds_american",
                           ])

     for state in STATES:
          df[f"{state}_prob_spread"] = df[f"sportsbook_{state}_prob"] - df[f"Kalshi_{state}_prob"]

     df["start_time_utc"] = pd.to_datetime(df["start_time_utc"])

     df = df.sort_values(by="start_time_utc", ascending=False)

     return df

In [97]:
tmp = read_season_data_file("nfl", 2025)
tmp = clean_data(tmp)

In [98]:
# Read and clean all sports for 2025 season
raw = {}
clean = {}

for sport in SPORTS_SEASONS.keys():
    raw[sport] = read_season_data_file(sport, 2025)
    clean[sport] = clean_data(raw[sport])
    print(len(clean[sport]))
    display(clean[sport].head())

277


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_home_open_interest,kalshi_away_volume,kalshi_away_open_interest,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
4,Chicago Bears,Los Angeles Rams,2026-01-18 23:30:00+00:00,17,20,189690.0,10309404.0,102558.0,3602778.0,292248.0,0.645,0.355,0.650625,0.349375,0.005625,-0.005625
1,New England Patriots,Houston Texans,2026-01-18 20:00:00+00:00,28,16,11354.0,2113013.0,35912.0,3765439.0,47266.0,0.375,0.635,0.378989,0.621011,0.003989,-0.013989
3,Denver Broncos,Buffalo Bills,2026-01-17 21:30:00+00:00,33,30,99259.0,6140810.0,135805.0,8737661.0,235064.0,0.465,0.535,0.466061,0.533939,0.001061,-0.001061
9,Pittsburgh Steelers,Houston Texans,2026-01-13 01:15:00+00:00,6,30,92237.0,7453726.0,70363.0,6888221.0,162600.0,0.595,0.405,0.585908,0.414092,-0.009092,0.009092
7,New England Patriots,Los Angeles Chargers,2026-01-12 01:15:00+00:00,16,3,86189.0,5171465.0,128976.0,5835078.0,215165.0,0.345,0.655,0.365101,0.634899,0.020101,-0.020101


665


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_home_open_interest,kalshi_away_volume,kalshi_away_open_interest,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,Toronto Blue Jays,Los Angeles Dodgers,2025-11-01 00:00:00+00:00,1.0,3.0,9526.0,2326441.0,101107.0,3573878.0,110633.0,0.565,0.435,0.565363,0.434637,0.000363,-0.000363
1,Los Angeles Dodgers,Toronto Blue Jays,2025-10-30 00:00:00+00:00,1.0,6.0,54742.0,1915331.0,5664.0,1897435.0,60406.0,0.355,0.645,0.349375,0.650625,-0.005625,0.005625
2,Los Angeles Dodgers,Toronto Blue Jays,2025-10-29 00:00:00+00:00,2.0,6.0,12994.0,1883129.0,19302.0,1571725.0,32296.0,0.355,0.645,0.349375,0.650625,-0.005625,0.005625
3,Los Angeles Dodgers,Toronto Blue Jays,2025-10-28 00:00:00+00:00,6.0,5.0,13586.0,2066408.0,3276.0,1000559.0,16862.0,0.355,0.645,0.355039,0.644961,0.000039,-0.000039
4,Toronto Blue Jays,Los Angeles Dodgers,2025-10-26 00:00:00+00:00,1.0,5.0,6287.0,988720.0,5819.0,2606347.0,12106.0,0.555,0.440,0.565363,0.434637,0.010363,-0.005363


1180


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_home_open_interest,kalshi_away_volume,kalshi_away_open_interest,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,San Antonio Spurs,New York Knicks,2026-06-14 00:30:00+00:00,90.0,94.0,138527.57,8194512.33,268615.60,29374803.78,407143.17,0.365,0.645,0.355039,0.644961,-0.009961,-0.000039
1,New York Knicks,San Antonio Spurs,2026-06-11 00:30:00+00:00,107.0,106.0,148522.77,11026484.24,152017.60,13981988.97,300540.37,0.465,0.545,0.445237,0.554763,-0.019763,0.009763
2,New York Knicks,San Antonio Spurs,2026-06-09 00:30:00+00:00,111.0,115.0,98582.65,11610517.26,188022.49,18522052.63,286605.14,0.475,0.525,0.466061,0.533939,-0.008939,0.008939
3,San Antonio Spurs,New York Knicks,2026-06-06 00:30:00+00:00,104.0,105.0,35197.11,6723748.92,166617.94,16136130.09,201815.05,0.315,0.685,0.317028,0.682972,0.002028,-0.002028
4,San Antonio Spurs,New York Knicks,2026-06-04 00:30:00+00:00,95.0,105.0,59303.66,7517888.05,144635.50,11530456.70,203939.16,0.365,0.635,0.355039,0.644961,-0.009961,0.009961


1091


,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_home_open_interest,kalshi_away_volume,kalshi_away_open_interest,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,Vegas Golden Knights,Carolina Hurricanes,2026-06-15 00:00:00+00:00,0.0,3.0,51541.43,1844420.91,13370.39,1788264.28,64911.82,0.525,0.485,0.511370,0.488630,-0.013630,0.003630
1,Carolina Hurricanes,Vegas Golden Knights,2026-06-12 00:00:00+00:00,4.0,2.0,8496.86,959245.68,63315.01,1970280.57,71811.87,0.405,0.595,0.414092,0.585908,0.009092,-0.009092
2,Vegas Golden Knights,Carolina Hurricanes,2026-06-10 00:00:00+00:00,3.0,5.0,18936.48,1432198.78,14659.86,1834093.65,33596.34,0.515,0.485,0.511370,0.488630,-0.003630,0.003630
3,Vegas Golden Knights,Carolina Hurricanes,2026-06-07 00:00:00+00:00,5.0,4.0,70448.68,1861317.94,29760.76,1618762.03,100209.44,0.505,0.495,0.500000,0.500000,-0.005000,0.005000
4,Carolina Hurricanes,Vegas Golden Knights,2026-06-05 00:00:00+00:00,4.0,3.0,24634.37,2172551.57,15634.68,957569.95,40269.05,0.395,0.605,0.405664,0.594336,0.010664,-0.010664


In [99]:
df_clean = pd.concat(clean.values())
df_clean = df_clean.sort_values(by="start_time_utc", ascending=True)
df_clean = df_clean.reset_index(drop=True)
df_clean

,home_team,away_team,start_time_utc,home_score,away_score,kalshi_home_volume,kalshi_home_open_interest,kalshi_away_volume,kalshi_away_open_interest,total_volume,Kalshi_away_prob,Kalshi_home_prob,sportsbook_away_prob,sportsbook_home_prob,away_prob_spread,home_prob_spread
0,Minnesota Twins,New York Mets,2025-04-16 17:10:00+00:00,4.0,3.0,1.00,1.00,49.00,331.00,50.00,0.470,0.520,0.477226,0.522774,0.007226,0.002774
1,Milwaukee Brewers,Detroit Tigers,2025-04-16 17:10:00+00:00,5.0,1.0,1.00,3.00,22.00,183.00,23.00,0.415,0.575,0.422880,0.577120,0.007880,0.002120
2,Philadelphia Phillies,San Francisco Giants,2025-04-16 22:45:00+00:00,4.0,11.0,170.00,420.00,32.00,476.00,202.00,0.420,0.565,0.432908,0.567092,0.012908,0.002092
3,Texas Rangers,Los Angeles Angels,2025-04-17 00:05:00+00:00,3.0,1.0,314.00,415.00,40.00,462.00,354.00,0.545,0.455,0.552919,0.447081,0.007919,-0.007919
4,Pittsburgh Pirates,Washington Nationals,2025-04-17 16:35:00+00:00,1.0,0.0,2.00,1724.00,122.00,800.00,124.00,0.425,0.580,0.422880,0.577120,-0.002120,-0.002880
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3208,Vegas Golden Knights,Carolina Hurricanes,2026-06-10 00:00:00+00:00,3.0,5.0,18936.48,1432198.78,14659.86,1834093.65,33596.34,0.515,0.485,0.511370,0.488630,-0.003630,0.003630
3209,New York Knicks,San Antonio Spurs,2026-06-11 00:30:00+00:00,107.0,106.0,148522.77,11026484.24,152017.60,13981988.97,300540.37,0.465,0.545,0.445237,0.554763,-0.019763,0.009763
3210,Carolina Hurricanes,Vegas Golden Knights,2026-06-12 00:00:00+00:00,4.0,2.0,8496.86,959245.68,63315.01,1970280.57,71811.87,0.405,0.595,0.414092,0.585908,0.009092,-0.009092
3211,San Antonio Spurs,New York Knicks,2026-06-14 00:30:00+00:00,90.0,94.0,138527.57,8194512.33,268615.60,29374803.78,407143.17,0.365,0.645,0.355039,0.644961,-0.009961,-0.000039


In [100]:
print(df_clean.dtypes)

home_team                                    str
away_team                                    str
start_time_utc               datetime64[us, UTC]
home_score                               float64
away_score                               float64
kalshi_home_volume                       float64
kalshi_home_open_interest                float64
kalshi_away_volume                       float64
kalshi_away_open_interest                float64
total_volume                             float64
Kalshi_away_prob                         float64
Kalshi_home_prob                         float64
sportsbook_away_prob                     float64
sportsbook_home_prob                     float64
away_prob_spread                         float64
home_prob_spread                         float64
dtype: object


In [105]:
df_clean.to_csv("../data/clean_data.csv")

In [101]:
# Inspecting low volume values...
for s, df in raw.items():
    v = df["kalshi_home_volume"] + df["kalshi_away_volume"]
    print(s, v.describe(percentiles=[.01, .05, .25, .5]).round(0).to_dict())
    print("  combined volume < 100:", (v < 100).sum(), "of", v.notna().sum())

nfl {'count': 279.0, 'mean': 52257.0, 'std': 54242.0, 'min': 487.0, '1%': 1310.0, '5%': 2974.0, '25%': 14202.0, '50%': 37742.0, 'max': 324346.0}
  combined volume < 100: 0 of 279
mlb {'count': 2150.0, 'mean': 1379.0, 'std': 6138.0, 'min': 0.0, '1%': 0.0, '5%': 0.0, '25%': 0.0, '50%': 90.0, 'max': 110633.0}
  combined volume < 100: 1107 of 2150
nba {'count': 1320.0, 'mean': 16918.0, 'std': 33497.0, 'min': 0.0, '1%': 141.0, '5%': 501.0, '25%': 2085.0, '50%': 6039.0, 'max': 483090.0}
  combined volume < 100: 12 of 1320
nhl {'count': 1393.0, 'mean': 3023.0, 'std': 8743.0, 'min': 0.0, '1%': 0.0, '5%': 23.0, '25%': 251.0, '50%': 732.0, 'max': 145017.0}
  combined volume < 100: 183 of 1393


In [102]:
df = raw["nba"].copy()
df["vol"] = df["kalshi_home_volume"] + df["kalshi_away_volume"]
cols = ["start_time_utc", "home_team", "away_team",
        "kalshi_home_ticker", "kalshi_away_ticker",
        "kalshi_home_candle_end_ts", "vol",
        "kalshi_home_yes_bid_close", "kalshi_home_yes_ask_close",
        "kalshi_away_yes_bid_close", "kalshi_away_yes_ask_close"]
df.nsmallest(15, "vol")[cols]

,start_time_utc,home_team,away_team,kalshi_home_ticker,kalshi_away_ticker,kalshi_home_candle_end_ts,vol,kalshi_home_yes_bid_close,kalshi_home_yes_ask_close,kalshi_away_yes_bid_close,kalshi_away_yes_ask_close
5,2026-05-31T00:00Z,Oklahoma City Thunder,San Antonio Spurs,KXNBAGAME-26MAY30SASOKC-OKC,KXNBAGAME-26MAY30SASOKC-SAS,1780185600,0.0,0.56,0.57,0.42,0.43
1181,2025-11-09T02:00Z,Denver Nuggets,Indiana Pacers,KXNBAGAME-25NOV08INDDEN-DEN,KXNBAGAME-25NOV08INDDEN-IND,1762653600,18.0,0.80,0.81,0.19,0.20
159,2026-04-04T00:00Z,Houston Rockets,Utah Jazz,KXNBAGAME-26APR03UTAHOU-HOU,KXNBAGAME-26APR03UTAHOU-UTA,1775260800,37.0,0.93,0.94,0.06,0.07
923,2025-12-19T01:00Z,Milwaukee Bucks,Toronto Raptors,KXNBAGAME-25DEC18TORMIL-MIL,KXNBAGAME-25DEC18TORMIL-TOR,1766106000,40.0,0.37,0.38,0.62,0.64
1316,2025-10-23T00:00Z,Milwaukee Bucks,Washington Wizards,KXNBAGAME-25OCT22WASMIL-MIL,KXNBAGAME-25OCT22WASMIL-WAS,1761177600,50.0,0.79,0.80,0.20,0.21
909,2025-12-21T00:00Z,New Orleans Pelicans,Indiana Pacers,KXNBAGAME-25DEC20INDNOP-NOP,KXNBAGAME-25DEC20INDNOP-IND,1766275200,51.0,0.56,0.57,0.43,0.45
740,2026-01-12T01:00Z,Phoenix Suns,Washington Wizards,KXNBAGAME-26JAN11WASPHX-PHX,KXNBAGAME-26JAN11WASPHX-WAS,1768179600,60.0,0.86,0.88,0.13,0.14
139,2026-04-07T23:00Z,Indiana Pacers,Minnesota Timberwolves,KXNBAGAME-26APR07MININD-IND,KXNBAGAME-26APR07MININD-MIN,1775602800,82.0,0.13,0.14,0.86,0.87
194,2026-03-31T00:00Z,Memphis Grizzlies,Phoenix Suns,KXNBAGAME-26MAR30PHXMEM-MEM,KXNBAGAME-26MAR30PHXMEM-PHX,1774915200,85.0,0.13,0.14,0.86,0.87
350,2026-03-11T02:00Z,Sacramento Kings,Indiana Pacers,KXNBAGAME-26MAR10INDSAC-SAC,KXNBAGAME-26MAR10INDSAC-IND,1773194400,88.0,0.59,0.60,0.41,0.42


In [103]:
ts = pd.to_datetime(df["kalshi_home_candle_end_ts"], unit="s", utc=True)
gap = (pd.to_datetime(df["start_time_utc"]) - ts).dt.total_seconds() / 60
gap.describe()

count    1320.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64

In [104]:
df.groupby(pd.to_datetime(df["start_time_utc"]).dt.to_period("M"))["vol"].median()

C:\Users\harri\AppData\Local\Temp\ipykernel_2756\2024590495.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df.groupby(pd.to_datetime(df["start_time_utc"]).dt.to_period("M"))["vol"].median()


start_time_utc
2025-10      4368.00
2025-11      3679.00
2025-12      4858.00
2026-01      5970.50
2026-02      7158.50
2026-03      5815.00
2026-04     12709.00
2026-05     57931.25
2026-06    286605.14
Freq: M, Name: vol, dtype: float64